# Bài 07: Hiệu chỉnh và Học nhạy cảm với chi phí (Calibration & Cost-Sensitive Learning)

**Mục tiêu bài học:**
- Hiểu khái niệm "hiệu chỉnh" (calibration) của mô hình và tại sao xác suất dự đoán đáng tin cậy lại quan trọng.
- Học cách đánh giá độ hiệu chỉnh của mô hình bằng biểu đồ hiệu chỉnh (Calibration Plot).
- Nắm được ý tưởng của Cost-Sensitive Learning: đưa "chi phí" của việc phân loại sai vào quá trình huấn luyện.
- Áp dụng Cost-Sensitive Learning cho các thuật toán hỗ trợ như Logistic Regression và Tree-based models thông qua tham số `class_weight`.

## 1. Vượt ra ngoài Nhãn dự đoán: Tầm quan trọng của Xác suất

Từ đầu đến giờ, chúng ta chủ yếu tập trung vào các nhãn dự đoán (0 hay 1) và các chỉ số dựa trên chúng (Precision, Recall, etc.). Nhưng hầu hết các mô hình phân loại không chỉ đưa ra nhãn, mà còn đưa ra một **xác suất dự đoán** (ví dụ: `predict_proba`).

Xác suất này cực kỳ quan trọng trong thực tế:
- **Ra quyết định kinh doanh:** Một khách hàng có 80% khả năng rời bỏ sẽ được ưu tiên chăm sóc hơn một khách hàng chỉ có 20% khả năng.
- **Xếp hạng:** Chúng ta có thể xếp hạng các ca nghi ngờ gian lận từ cao đến thấp dựa trên xác suất và chỉ điều tra top N ca có xác suất cao nhất.
- **Kết hợp mô hình:** Xác suất từ nhiều mô hình có thể được kết hợp với nhau.

**Câu hỏi đặt ra là:** Liệu xác suất mà mô hình đưa ra có "thật" không? Nếu mô hình dự đoán 100 mẫu có xác suất 80% là lớp 1, thì liệu có thực sự khoảng 80 mẫu trong số đó thuộc lớp 1 không? Một mô hình đáp ứng được điều này được gọi là **well-calibrated (được hiệu chỉnh tốt)**.

### 1.1. Biểu đồ hiệu chỉnh (Calibration Plot)

Đây là công cụ trực quan để đánh giá độ hiệu chỉnh của mô hình.

**Cách đọc biểu đồ:**
- **Trục x:** Xác suất dự đoán trung bình trong một khoảng (bin). Ví dụ, tất cả các dự đoán có xác suất từ 0.1 đến 0.2 sẽ được gom vào một bin.
- **Trục y:** Tỷ lệ thực tế của lớp dương tính trong bin đó. Ví dụ, trong các mẫu có xác suất dự đoán từ 0.1-0.2, tỷ lệ thực tế là lớp 1 là bao nhiêu.
- **Đường chéo hoàn hảo (Perfectly calibrated):** Là đường thẳng `y=x`. Nếu đường cong của mô hình nằm càng gần đường này, mô hình càng được hiệu chỉnh tốt.
- **Đường cong của mô hình:** Nếu đường cong nằm dưới đường chéo, mô hình đang **quá tự tin (over-confident)**. Ví dụ, nó dự đoán 80% nhưng thực tế chỉ có 60% là lớp 1. Nếu đường cong nằm trên đường chéo, mô hình đang **thiếu tự tin (under-confident)**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import calibration_curve, CalibrationDisplay

# Tạo dữ liệu
X, y = make_classification(n_samples=10000, n_features=20, n_informative=10, n_redundant=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.90, random_state=42)

# Huấn luyện 2 mô hình: Logistic Regression (thường được hiệu chỉnh tốt) và Naive Bayes (thường bị hiệu chỉnh kém)
lr = LogisticRegression(random_state=42)
gnb = GaussianNB()

lr.fit(X_train, y_train)
gnb.fit(X_train, y_train)

# Lấy xác suất dự đoán cho lớp 1
prob_lr = lr.predict_proba(X_test)[:, 1]
prob_gnb = gnb.predict_proba(X_test)[:, 1]

# Vẽ biểu đồ
fig, ax = plt.subplots(figsize=(10, 7))
disp_lr = CalibrationDisplay.from_predictions(y_test, prob_lr, n_bins=10, name='Logistic Regression', ax=ax)
disp_gnb = CalibrationDisplay.from_predictions(y_test, prob_gnb, n_bins=10, name='Gaussian Naive Bayes', ax=ax)
ax.set_title('Biểu đồ hiệu chỉnh (Calibration Plot)')
plt.show()

**Nhận xét:**
Đường cong của `Logistic Regression` bám rất sát đường chéo, cho thấy nó được hiệu chỉnh tốt. Trong khi đó, đường cong của `Gaussian Naive Bayes` có dạng hình chữ S, một dấu hiệu điển hình của việc hiệu chỉnh kém: nó quá tự tin với các dự đoán ở hai đầu (gần 0 và 1) và thiếu tự tin ở khoảng giữa.

**Lưu ý:** Các kỹ thuật resampling như SMOTE có thể làm hỏng độ hiệu chỉnh của mô hình. Sau khi dùng SMOTE, xác suất dự đoán có thể không còn phản ánh đúng thực tế nữa. Việc hiệu chỉnh lại mô hình (ví dụ dùng `CalibratedClassifierCV` của Scikit-learn) là một bước nâng cao cần thiết nếu xác suất đáng tin cậy là quan trọng.

## 2. Học nhạy cảm với chi phí (Cost-Sensitive Learning)

Resampling là một cách tiếp cận **dựa trên dữ liệu**. Một cách tiếp cận khác là **dựa trên thuật toán**. Thay vì thay đổi dữ liệu, chúng ta "dạy" cho thuật toán biết rằng việc phân loại sai các lớp khác nhau sẽ có "chi phí" (cost) khác nhau.

**Ý tưởng:** Trong quá trình huấn luyện, khi tính toán hàm mất mát (loss function), chúng ta sẽ nhân lỗi của mỗi mẫu với một trọng số (weight). Nếu một mẫu thuộc lớp thiểu số bị phân loại sai, lỗi của nó sẽ được nhân với một trọng số lớn hơn. Điều này tạo ra một "hình phạt" nặng hơn, buộc mô hình phải nỗ lực hơn để phân loại đúng các mẫu của lớp thiểu số.

Nhiều thuật toán trong Scikit-learn hỗ trợ phương pháp này thông qua tham số `class_weight`.

### 2.1. `class_weight='balanced'`

Đây là cách sử dụng đơn giản và phổ biến nhất. Khi bạn đặt `class_weight='balanced'`, Scikit-learn sẽ tự động tính toán các trọng số tỷ lệ nghịch với tần suất của mỗi lớp.

**Công thức:**
$$ w_j = \frac{n_{\text{samples}}}{n_{\text{classes}} \times n_{\text{samples}_j}} $$

Trong đó:
- $w_j$ là trọng số của lớp $j$.
- $n_{\text{samples}}$ là tổng số mẫu.
- $n_{\text{classes}}$ là tổng số lớp.
- $n_{\text{samples}_j}$ là số mẫu của lớp $j$.

Lớp nào càng hiếm (số mẫu ít), trọng số của nó sẽ càng cao.

In [ ]:
from collections import Counter
from sklearn.metrics import classification_report, balanced_accuracy_score

# Tạo lại dữ liệu mất cân bằng
X, y = make_classification(n_samples=10000, weights=[0.98, 0.02], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Phân phối lớp trong tập huấn luyện: {Counter(y_train)}")

# 1. Mô hình gốc (không xử lý)
model_base = LogisticRegression(random_state=42)
model_base.fit(X_train, y_train)
y_pred_base = model_base.predict(X_test)

print("\n--- 1. Kết quả mô hình GỐC ---")
print(classification_report(y_test, y_pred_base))
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_base):.4f}")

# 2. Mô hình với class_weight='balanced'
model_weighted = LogisticRegression(random_state=42, class_weight='balanced')
model_weighted.fit(X_train, y_train)
y_pred_weighted = model_weighted.predict(X_test)

print("\n--- 2. Kết quả mô hình với class_weight='balanced' ---")
print(classification_report(y_test, y_pred_weighted))
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_weighted):.4f}")

**Nhận xét:**
Chỉ bằng cách thêm một tham số `class_weight='balanced'`, kết quả đã cải thiện một cách đáng kinh ngạc. `Recall` của lớp 1 đã tăng vọt (ví dụ từ 0.18 lên 0.82), trong khi `Recall` của lớp 0 chỉ giảm nhẹ. `Balanced Accuracy` cũng tăng mạnh.

Đây là một phương pháp cực kỳ hiệu quả, đơn giản và nhanh chóng.

### 2.2. Tự định nghĩa `class_weight`

Ngoài việc sử dụng `'balanced'`, bạn cũng có thể tự định nghĩa một dictionary cho `class_weight` để tinh chỉnh mức độ "phạt". Điều này cho phép bạn kiểm soát chính xác hơn sự đánh đổi giữa Precision và Recall.

Ví dụ, nếu bạn muốn "phạt" việc bỏ sót lớp 1 nặng hơn nữa, bạn có thể tăng trọng số của nó lên.

In [ ]:
# Tự định nghĩa trọng số: phạt lớp 1 gấp 20 lần lớp 0
weights = {0: 1, 1: 20}

model_custom_weight = LogisticRegression(random_state=42, class_weight=weights)
model_custom_weight.fit(X_train, y_train)
y_pred_custom = model_custom_weight.predict(X_test)

print("\n--- 3. Kết quả mô hình với class_weight tùy chỉnh {0:1, 1:20} ---")
print(classification_report(y_test, y_pred_custom))
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_custom):.4f}")

**Nhận xét:**
Bằng cách tăng trọng số của lớp 1, chúng ta có thể đẩy `Recall` của lớp 1 lên cao hơn nữa, nhưng thường sẽ phải trả giá bằng việc `Precision` của lớp 1 và `Recall` của lớp 0 giảm xuống. Việc tìm ra bộ trọng số tối ưu thường đòi hỏi phải thử nghiệm và tinh chỉnh (ví dụ: dùng Grid Search).

## 3. So sánh Resampling và Cost-Sensitive Learning

| Phương pháp             | Ưu điểm                                                                      | Nhược điểm                                                              | Khi nào nên dùng?                                                                                             |
|--------------------------|------------------------------------------------------------------------------|-------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------|
| **Resampling (SMOTE)**   | Hiệu quả cao, hoạt động với mọi loại thuật toán.                              | Có thể làm hỏng độ hiệu chỉnh của xác suất, tăng thời gian huấn luyện.   | Là lựa chọn mặc định tốt. Đặc biệt hữu ích khi thuật toán không hỗ trợ `class_weight`.                         |
| **Cost-Sensitive (class_weight)** | **Rất nhanh**, đơn giản (chỉ cần thêm 1 tham số), không thay đổi dữ liệu gốc. | Chỉ một số thuật toán hỗ trợ (Logistic Regression, SVM, Trees, ...).     | **Hãy thử nó đầu tiên!** Nếu thuật toán của bạn hỗ trợ, đây là cách nhanh nhất và thường rất hiệu quả để có được một baseline mạnh. |

**Lời khuyên:**
Trong một dự án thực tế, quy trình làm việc hiệu quả thường là:
1.  Xây dựng mô hình baseline không xử lý gì cả.
2.  Thử ngay `class_weight='balanced'` nếu thuật toán hỗ trợ. Đây là cách nhanh nhất để xem liệu việc xử lý mất cân bằng có mang lại hiệu quả không.
3.  Thử nghiệm với SMOTE. So sánh kết quả của SMOTE với `class_weight`.
4.  Bạn cũng có thể kết hợp cả hai, nhưng cần cẩn thận. Đôi khi việc áp dụng cả hai có thể dẫn đến việc "bù trừ quá mức" (over-compensating).
5.  Chọn ra phương pháp cho kết quả tốt nhất trên tập kiểm tra dựa trên chỉ số đánh giá mà bạn đã chọn (ví dụ: Balanced Accuracy, F2-score).

## 4. Kết luận

Chúng ta đã học hai kỹ thuật nâng cao và mạnh mẽ: Calibration và Cost-Sensitive Learning. Calibration đảm bảo rằng kết quả xác suất của mô hình là đáng tin cậy, trong khi Cost-Sensitive Learning cung cấp một giải pháp thay thế hiệu quả và nhanh chóng cho resampling.

Với kiến thức từ 7 bài học, bạn đã có một bộ công cụ toàn diện để đối mặt với các bài toán phân loại trên dữ liệu mất cân bằng trong thế giới thực. Từ việc chẩn đoán vấn đề, lựa chọn chỉ số đánh giá phù hợp, cho đến việc áp dụng các kỹ thuật xử lý ở cả cấp độ dữ liệu và thuật toán.

Trong bài học cuối cùng, chúng ta sẽ tổng hợp tất cả kiến thức này vào một **dự án cuối khóa**, mô phỏng một quy trình làm việc hoàn chỉnh từ đầu đến cuối.